In [1]:
from google.colab import files

uploaded = files.upload()

Saving road_traffic_sensor_data.csv to road_traffic_sensor_data.csv
Saving traffic_accident_data.csv to traffic_accident_data.csv


In [2]:
import pandas as pd
import numpy as np

In [3]:
traffic_df = pd.read_csv('/content/road_traffic_sensor_data.csv')

accident_df = pd.read_csv('/content/traffic_accident_data.csv')

In [4]:
traffic_df.head()

traffic_df.info()

traffic_df.isnull().sum()

accident_df.head()

accident_df.info()

accident_df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Sensor_ID         300 non-null    object
 1   Location          300 non-null    object
 2   Date_Time         300 non-null    object
 3   Vehicle_Count     300 non-null    int64 
 4   Average_Speed     300 non-null    int64 
 5   Congestion_Level  300 non-null    object
dtypes: int64(2), object(4)
memory usage: 14.2+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Accident_ID         5000 non-null   object
 1   Date_Time           5000 non-null   object
 2   Location            5000 non-null   object
 3   Weather_Condition   5000 non-null   object
 4   Road_Condition      5000 non-null   object
 5   Vehicle_Type        5000 non-

,0
Accident_ID,0
Date_Time,0
Location,0
Weather_Condition,0
Road_Condition,0
Vehicle_Type,0
Accident_Severity,0
Number_of_Vehicles,0
Casualties,0
Traffic_Density,0


In [5]:
traffic_df.drop_duplicates(inplace=True)

accident_df.drop_duplicates(inplace=True)

In [6]:
traffic_df['date_time'] = pd.to_datetime(traffic_df['date_time'])

accident_df['date_time'] = pd.to_datetime(accident_df['date_time'])

KeyError: 'date_time'

In [7]:
print(traffic_df.columns)

Index(['Sensor_ID', 'Location', 'Date_Time', 'Vehicle_Count', 'Average_Speed',
       'Congestion_Level'],
      dtype='object')


In [8]:
traffic_df['Date_Time'] = pd.to_datetime(traffic_df['Date_Time'])

accident_df['Date_Time'] = pd.to_datetime(accident_df['Date_Time'])

In [9]:
traffic_df['date'] = traffic_df['date_time'].dt.date
traffic_df['day'] = traffic_df['date_time'].dt.day_name()
traffic_df['month'] = traffic_df['date_time'].dt.month_name()
traffic_df['year'] = traffic_df['date_time'].dt.year

accident_df['date'] = accident_df['date_time'].dt.date
accident_df['day'] = accident_df['date_time'].dt.day_name()
accident_df['month'] = accident_df['date_time'].dt.month_name()
accident_df['year'] = accident_df['date_time'].dt.year

KeyError: 'date_time'

In [10]:
traffic_df['Date'] = traffic_df['Date_Time'].dt.date
traffic_df['Day'] = traffic_df['Date_Time'].dt.day_name()
traffic_df['Month'] = traffic_df['Date_Time'].dt.month_name()
traffic_df['Year'] = traffic_df['Date_Time'].dt.year

accident_df['Date'] = accident_df['Date_Time'].dt.date
accident_df['Day'] = accident_df['Date_Time'].dt.day_name()
accident_df['Month'] = accident_df['Date_Time'].dt.month_name()
accident_df['Year'] = accident_df['Date_Time'].dt.year

In [11]:
traffic_df['Hour'] = traffic_df['Date_Time'].dt.hour

accident_df['Hour'] = accident_df['Date_Time'].dt.hour

In [12]:
traffic_df['Peak_Status'] = np.where(
    (traffic_df['Hour'].between(7,10)) |
    (traffic_df['Hour'].between(17,20)),
    'Peak Hour',
    'Non Peak Hour'
)

In [13]:
accident_df['Accident_Severity'].unique()

array(['Minor', 'Moderate', 'Fatal', 'Severe'], dtype=object)

In [14]:
severity_map = {
    'Minor':1,
    'Moderate':2,
    'Severe':3,
    'Fatal':4
}

accident_df['Severity_Score'] = accident_df['Accident_Severity'].map(severity_map)

In [15]:
accident_df['Accident_Risk_Score'] = (
    accident_df['Severity_Score']
    + accident_df['Casualties']
)

In [16]:
print(traffic_df.columns)

Index(['Sensor_ID', 'Location', 'Date_Time', 'Vehicle_Count', 'Average_Speed',
       'Congestion_Level', 'Date', 'Day', 'Month', 'Year', 'Hour',
       'Peak_Status'],
      dtype='object')


In [17]:
print(accident_df.columns)

Index(['Accident_ID', 'Date_Time', 'Location', 'Weather_Condition',
       'Road_Condition', 'Vehicle_Type', 'Accident_Severity',
       'Number_of_Vehicles', 'Casualties', 'Traffic_Density', 'Date', 'Day',
       'Month', 'Year', 'Hour', 'Severity_Score', 'Accident_Risk_Score'],
      dtype='object')


In [18]:
dim_date = pd.concat([
    traffic_df[['Date','Day','Month','Year']],
    accident_df[['Date','Day','Month','Year']]
]).drop_duplicates().reset_index(drop=True)

dim_date['Date_Key'] = range(1, len(dim_date)+1)

dim_date = dim_date[
    ['Date_Key','Date','Day','Month','Year']
]

dim_date.head()

,Date_Key,Date,Day,Month,Year
0,1,2024-01-01,Monday,January,2024
1,2,2024-01-02,Tuesday,January,2024
2,3,2024-01-03,Wednesday,January,2024
3,4,2024-01-04,Thursday,January,2024
4,5,2024-01-05,Friday,January,2024


In [19]:
dim_time = pd.DataFrame(
    traffic_df[['Hour','Peak_Status']]
).drop_duplicates()

dim_time['Time_Key'] = range(1,len(dim_time)+1)

dim_time = dim_time[
    ['Time_Key','Hour','Peak_Status']
]

dim_time.head()

,Time_Key,Hour,Peak_Status
0,1,0,Non Peak Hour
1,2,1,Non Peak Hour
2,3,2,Non Peak Hour
3,4,3,Non Peak Hour
4,5,4,Non Peak Hour


In [20]:
dim_weather = accident_df[
    ['Weather_Condition']
].drop_duplicates()

dim_weather['Weather_Key'] = range(
    1,
    len(dim_weather)+1
)

dim_weather = dim_weather[
    ['Weather_Key','Weather_Condition']
]

dim_weather.head()

,Weather_Key,Weather_Condition
0,1,Fog
1,2,Clear
2,3,Snow
4,4,Storm
8,5,Rain


In [21]:
dim_road = accident_df[
    ['Road_Condition']
].drop_duplicates()

dim_road['Road_Key'] = range(
    1,
    len(dim_road)+1
)

dim_road = dim_road[
    ['Road_Key','Road_Condition']
]

dim_road.head()

,Road_Key,Road_Condition
0,1,Icy
2,2,Dry
4,3,Under Construction
5,4,Wet


In [22]:
dim_vehicle = accident_df[
    ['Vehicle_Type']
].drop_duplicates()

dim_vehicle['Vehicle_Key'] = range(
    1,
    len(dim_vehicle)+1
)

dim_vehicle = dim_vehicle[
    ['Vehicle_Key','Vehicle_Type']
]

dim_vehicle.head()

,Vehicle_Key,Vehicle_Type
0,1,Bus
2,2,Motorcycle
3,3,Bicycle
5,4,Car
16,5,Truck


In [23]:
dim_location = pd.concat([
    traffic_df[['Location']],
    accident_df[['Location']]
]).drop_duplicates()

dim_location['Location_Key'] = range(
    1,
    len(dim_location)+1
)

dim_location = dim_location[
    ['Location_Key','Location']
]

dim_location.head()

,Location_Key,Location
0,1,Downtown
2,2,Highway
4,3,Residential Zone
7,4,Industrial Area
9,5,Suburbs


In [24]:
fact_traffic = traffic_df.copy()

In [25]:
fact_traffic = fact_traffic.merge(
    dim_location,
    on='Location',
    how='left'
)

In [26]:
fact_traffic = fact_traffic.merge(
    dim_date,
    on=['Date','Day','Month','Year'],
    how='left'
)

In [27]:
fact_traffic = fact_traffic.merge(
    dim_time,
    on=['Hour','Peak_Status'],
    how='left'
)

In [28]:
fact_traffic = fact_traffic[
[
'Date_Key',
'Time_Key',
'Location_Key',
'Vehicle_Count',
'Average_Speed',
'Congestion_Index'
]
]

fact_traffic.head()

KeyError: "['Congestion_Index'] not in index"

In [29]:
print(fact_traffic.columns.tolist())

['Sensor_ID', 'Location', 'Date_Time', 'Vehicle_Count', 'Average_Speed', 'Congestion_Level', 'Date', 'Day', 'Month', 'Year', 'Hour', 'Peak_Status', 'Location_Key', 'Date_Key', 'Time_Key']


In [30]:
print(fact_traffic.columns)

Index(['Sensor_ID', 'Location', 'Date_Time', 'Vehicle_Count', 'Average_Speed',
       'Congestion_Level', 'Date', 'Day', 'Month', 'Year', 'Hour',
       'Peak_Status', 'Location_Key', 'Date_Key', 'Time_Key'],
      dtype='object')


In [31]:
fact_traffic = fact_traffic[
[
'Date_Key',
'Time_Key',
'Location_Key',
'vehicle_count',
'average_speed',
'congestion_index'
]
]

KeyError: "['vehicle_count', 'average_speed', 'congestion_index'] not in index"

In [32]:
print(fact_traffic.columns.tolist())

['Sensor_ID', 'Location', 'Date_Time', 'Vehicle_Count', 'Average_Speed', 'Congestion_Level', 'Date', 'Day', 'Month', 'Year', 'Hour', 'Peak_Status', 'Location_Key', 'Date_Key', 'Time_Key']


In [33]:
fact_traffic.head()

,Sensor_ID,Location,Date_Time,Vehicle_Count,Average_Speed,Congestion_Level,Date,Day,Month,Year,Hour,Peak_Status,Location_Key,Date_Key,Time_Key
0,a36f14,Downtown,2024-01-01 00:00:00,90,59,High,2024-01-01,Monday,January,2024,0,Non Peak Hour,1,1,1
1,256a39,Downtown,2024-01-01 01:00:00,324,73,Moderate,2024-01-01,Monday,January,2024,1,Non Peak Hour,1,1,2
2,a8a01f,Highway,2024-01-01 02:00:00,61,56,Moderate,2024-01-01,Monday,January,2024,2,Non Peak Hour,2,1,3
3,3036e3,Highway,2024-01-01 03:00:00,90,56,High,2024-01-01,Monday,January,2024,3,Non Peak Hour,2,1,4
4,2b45b1,Residential Zone,2024-01-01 04:00:00,72,77,High,2024-01-01,Monday,January,2024,4,Non Peak Hour,3,1,5


In [34]:
for col in fact_traffic.columns:
    print(col)

Sensor_ID
Location
Date_Time
Vehicle_Count
Average_Speed
Congestion_Level
Date
Day
Month
Year
Hour
Peak_Status
Location_Key
Date_Key
Time_Key


In [35]:
traffic_df.columns


Index(['Sensor_ID', 'Location', 'Date_Time', 'Vehicle_Count', 'Average_Speed',
       'Congestion_Level', 'Date', 'Day', 'Month', 'Year', 'Hour',
       'Peak_Status'],
      dtype='object')

In [36]:
traffic_df['Congestion_Index'] = (
    traffic_df['Vehicle_Count'] /
    traffic_df['Average_Speed']
).round(2)

In [37]:
traffic_df.head()

,Sensor_ID,Location,Date_Time,Vehicle_Count,Average_Speed,Congestion_Level,Date,Day,Month,Year,Hour,Peak_Status,Congestion_Index
0,a36f14,Downtown,2024-01-01 00:00:00,90,59,High,2024-01-01,Monday,January,2024,0,Non Peak Hour,1.53
1,256a39,Downtown,2024-01-01 01:00:00,324,73,Moderate,2024-01-01,Monday,January,2024,1,Non Peak Hour,4.44
2,a8a01f,Highway,2024-01-01 02:00:00,61,56,Moderate,2024-01-01,Monday,January,2024,2,Non Peak Hour,1.09
3,3036e3,Highway,2024-01-01 03:00:00,90,56,High,2024-01-01,Monday,January,2024,3,Non Peak Hour,1.61
4,2b45b1,Residential Zone,2024-01-01 04:00:00,72,77,High,2024-01-01,Monday,January,2024,4,Non Peak Hour,0.94


In [38]:
fact_traffic = traffic_df.copy()

In [39]:
fact_traffic = fact_traffic.merge(
    dim_location,
    on='Location',
    how='left'
)

In [40]:
fact_traffic = fact_traffic.merge(
    dim_date,
    on=['Date','Day','Month','Year'],
    how='left'
)

In [41]:
fact_traffic = fact_traffic.merge(
    dim_time,
    on=['Hour','Peak_Status'],
    how='left'
)

In [42]:
fact_traffic = fact_traffic[
[
'Date_Key',
'Time_Key',
'Location_Key',
'Vehicle_Count',
'Average_Speed',
'Congestion_Index'
]
]

In [43]:
print(accident_df.columns.tolist())

['Accident_ID', 'Date_Time', 'Location', 'Weather_Condition', 'Road_Condition', 'Vehicle_Type', 'Accident_Severity', 'Number_of_Vehicles', 'Casualties', 'Traffic_Density', 'Date', 'Day', 'Month', 'Year', 'Hour', 'Severity_Score', 'Accident_Risk_Score']


In [44]:
fact_accident = accident_df.copy()


In [45]:
fact_accident = fact_accident.merge(
    dim_location,
    on='Location',
    how='left'
)

In [46]:
fact_accident = fact_accident.merge(
    dim_date,
    on=['Date','Day','Month','Year'],
    how='left'
)

In [47]:
fact_accident = fact_accident.merge(
    dim_time[['Hour','Time_Key']],
    on='Hour',
    how='left'
)

In [48]:
fact_accident = fact_accident.merge(
    dim_weather,
    on='Weather_Condition',
    how='left'
)


In [49]:
fact_accident = fact_accident.merge(
    dim_road,
    on='Road_Condition',
    how='left'
)

In [50]:
fact_accident = fact_accident.merge(
    dim_vehicle,
    on='Vehicle_Type',
    how='left'
)

In [51]:
fact_accident = fact_accident[
[
'Date_Key',
'Time_Key',
'Location_Key',
'Weather_Key',
'Road_Key',
'Vehicle_Key',
'Severity_Score',
'Accident_Risk_Score',
'Casualties',
'Number_of_Vehicles'
]
]

In [52]:
dim_date.to_csv("dim_date.csv", index=False)

dim_time.to_csv("dim_time.csv", index=False)

dim_weather.to_csv("dim_weather.csv", index=False)

dim_road.to_csv("dim_road_condition.csv", index=False)

dim_vehicle.to_csv("dim_vehicle_type.csv", index=False)

dim_location.to_csv("dim_location.csv", index=False)

fact_traffic.to_csv("fact_traffic.csv", index=False)

fact_accident.to_csv("fact_accident.csv", index=False)

In [54]:
from google.colab import files

files.download('dim_date.csv')
files.download('dim_time.csv')
files.download('dim_weather.csv')
files.download('dim_road_condition.csv')
files.download('dim_vehicle_type.csv')
files.download('dim_location.csv')
files.download('fact_traffic.csv')
files.download('fact_accident.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [55]:
q1 = traffic_df['Congestion_Index'].quantile(0.33)
q2 = traffic_df['Congestion_Index'].quantile(0.66)

traffic_df['Congestion_Risk'] = np.where(
    traffic_df['Congestion_Index'] <= q1,
    'Low',
    np.where(
        traffic_df['Congestion_Index'] <= q2,
        'Medium',
        'High'
    )
)

In [56]:
traffic_df[['Congestion_Index','Congestion_Risk']].head()

,Congestion_Index,Congestion_Risk
0,1.53,Low
1,4.44,Medium
2,1.09,Low
3,1.61,Low
4,0.94,Low


In [57]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

In [58]:
X = traffic_df[
    ['Vehicle_Count',
     'Average_Speed',
     'Hour']
]

y = traffic_df['Congestion_Risk']

In [59]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [60]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [61]:
y_pred = rf_model.predict(X_test)

In [62]:
accuracy = accuracy_score(
    y_test,
    y_pred
)

print("Accuracy:", accuracy)

Accuracy: 0.9666666666666667


In [63]:
print(
    classification_report(
        y_test,
        y_pred
    )
)

              precision    recall  f1-score   support

        High       1.00      1.00      1.00        17
         Low       0.88      1.00      0.94        15
      Medium       1.00      0.93      0.96        28

    accuracy                           0.97        60
   macro avg       0.96      0.98      0.97        60
weighted avg       0.97      0.97      0.97        60



In [64]:
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
})

importance.sort_values(
    by='Importance',
    ascending=False
)

,Feature,Importance
0,Vehicle_Count,0.535632
1,Average_Speed,0.360764
2,Hour,0.103604
